# Chapter 13: Test-Time Compute: Scaling at Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part3_advanced/13_test_time_compute.ipynb)

> **Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar (Packt, 2025)  
> **Chapter 13:** Test-Time Compute: Scaling at Inference  
> **Notebook:** `part3_advanced/13_test_time_compute.ipynb`

---

## What this notebook covers

This is the companion notebook for Chapter 13 of the book. Run it on a free Colab T4 GPU. All code uses small, publicly available models (under 500 MB) that fit within the free tier memory limit.

**To open in Colab:** Click the badge above, or replace `github.com` with `githubtocolab.com` in the URL.


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q transformers==4.40.0 torch accelerate

## 1. Imports and Setup

In [ ]:
import re
import math
import random
import warnings
import collections
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
random.seed(3)
np.random.seed(3)
torch.manual_seed(3)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Load Qwen/Qwen2.5-0.5B-Instruct

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()
print(f'Loaded {MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

## 3. Decoding Strategies from Scratch

We implement all strategies by manipulating the raw logits from Qwen/Qwen2.5-0.5B-Instruct — no `model.generate()` magic box.

In [ ]:
def get_next_token_logits(prompt: str) -> torch.Tensor:
    """Return logits for the next token position."""
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model(**enc)
    return out.logits[0, -1, :]  # (V,)


def decode_greedy(prompt: str, max_new: int = 40) -> str:
    """Greedy decoding — always pick argmax token."""
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
    for _ in range(max_new):
        with torch.no_grad():
            logits = model(ids).logits[0, -1, :]
        next_id = torch.argmax(logits).unsqueeze(0).unsqueeze(0)
        ids = torch.cat([ids, next_id], dim=1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    new_ids = ids[0][tokenizer(prompt, return_tensors='pt').input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def decode_temperature(prompt: str, temp: float = 0.8, max_new: int = 40) -> str:
    """Temperature sampling."""
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
    for _ in range(max_new):
        with torch.no_grad():
            logits = model(ids).logits[0, -1, :] / max(temp, 1e-6)
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1).unsqueeze(0)
        ids = torch.cat([ids, next_id], dim=1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    new_ids = ids[0][tokenizer(prompt, return_tensors='pt').input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def decode_topk(prompt: str, k: int = 50, temp: float = 0.8, max_new: int = 40) -> str:
    """Top-k sampling."""
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
    for _ in range(max_new):
        with torch.no_grad():
            logits = model(ids).logits[0, -1, :]
        topk_vals, topk_idx = torch.topk(logits, k)
        probs = F.softmax(topk_vals / max(temp, 1e-6), dim=-1)
        chosen = topk_idx[torch.multinomial(probs, 1)]
        ids = torch.cat([ids, chosen.unsqueeze(0).unsqueeze(0)], dim=1)
        if chosen.item() == tokenizer.eos_token_id:
            break
    new_ids = ids[0][tokenizer(prompt, return_tensors='pt').input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def decode_topp(prompt: str, p: float = 0.9, temp: float = 0.8, max_new: int = 40) -> str:
    """Nucleus (top-p) sampling."""
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
    for _ in range(max_new):
        with torch.no_grad():
            logits = model(ids).logits[0, -1, :] / max(temp, 1e-6)
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cum_probs = torch.cumsum(sorted_probs, dim=0)
        cutoff = (cum_probs > p).nonzero(as_tuple=True)[0]
        cutoff_idx = cutoff[0].item() + 1 if len(cutoff) > 0 else len(sorted_probs)
        nucleus_probs = sorted_probs[:cutoff_idx]
        nucleus_idx   = sorted_idx[:cutoff_idx]
        chosen_pos = torch.multinomial(nucleus_probs / nucleus_probs.sum(), 1)
        chosen = nucleus_idx[chosen_pos]
        ids = torch.cat([ids, chosen.unsqueeze(0).unsqueeze(0)], dim=1)
        if chosen.item() == tokenizer.eos_token_id:
            break
    new_ids = ids[0][tokenizer(prompt, return_tensors='pt').input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


print('Decoding strategies defined.')

### 3a. Output Diversity Comparison

In [ ]:
PROMPT = 'The best way to learn machine learning is'
N_SHOW = 3

strategies = [
    ('Greedy',       lambda: decode_greedy(PROMPT, max_new=30)),
    ('Temp=0.5',     lambda: decode_temperature(PROMPT, temp=0.5, max_new=30)),
    ('Temp=1.2',     lambda: decode_temperature(PROMPT, temp=1.2, max_new=30)),
    ('Top-k k=10',   lambda: decode_topk(PROMPT, k=10,  temp=0.8, max_new=30)),
    ('Top-p p=0.9',  lambda: decode_topp(PROMPT, p=0.9, temp=0.8, max_new=30)),
]

for name, fn in strategies:
    print(f'\n=== {name} ===')
    outputs = set()
    for _ in range(N_SHOW):
        o = fn()[:80]
        outputs.add(o)
        print(f'  > {o}')
    print(f'  Unique responses: {len(outputs)}/{N_SHOW}')

## 4. Best-of-N: Theoretical vs Empirical

In [ ]:
def extract_answer(text: str):
    for pat in [r'(?:answer|result|=)\s*[:]?\s*(-?\d+\.?\d*)',
                r'(-?\d+\.?\d*)\s*$',
                r'(-?\d+\.?\d*)',]:
        m = re.search(pat, text.strip(), re.IGNORECASE)
        if m:
            try: return float(m.group(1))
            except ValueError: pass
    return None


EVAL_PROBLEMS = [
    ('What is 3 + 4?', 7), ('What is 10 - 3?', 7), ('What is 5 + 6?', 11),
    ('What is 8 - 2?', 6), ('What is 4 + 9?', 13), ('What is 15 - 7?', 8),
]

def reward(response: str, gold: float) -> float:
    pred = extract_answer(response)
    return 1.0 if (pred is not None and abs(pred - gold) <= 0.5) else 0.0


# Estimate single-sample accuracy
single_acc_list = []
for question, gold in EVAL_PROBLEMS:
    resp = decode_topp(f'Q: {question}\nA: The answer is', p=0.95, temp=0.8, max_new=20)
    single_acc_list.append(reward(resp, gold))
p_single = max(float(np.mean(single_acc_list)), 0.05)
print(f'Single-sample accuracy: p = {p_single:.3f}')

N_vals = [1, 2, 4, 8, 16]
theoretical = [1 - (1 - p_single)**n for n in N_vals]

empirical = []
for n in N_vals:
    hits = 0
    for question, gold in EVAL_PROBLEMS:
        prompt = f'Q: {question}\nA: The answer is'
        for _ in range(n):
            resp = decode_topp(prompt, p=0.95, temp=0.9, max_new=20)
            if reward(resp, gold) == 1.0:
                hits += 1
                break
    empirical.append(hits / len(EVAL_PROBLEMS))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(N_vals, theoretical, 'b-o', label=f'Theoretical 1-(1-p)^N  (p={p_single:.2f})')
ax.plot(N_vals, empirical,   'r--s', label='Empirical best-of-N')
ax.set_xscale('log', base=2); ax.set_xticks(N_vals); ax.set_xticklabels(N_vals)
ax.set_xlabel('N'); ax.set_ylabel('Accuracy (fraction with ≥1 correct)')
ax.set_title('Best-of-N Accuracy')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('best_of_n_ttc.png', dpi=120)
plt.show()

## 5. Self-Consistency — Majority Vote

In [ ]:
def majority_vote(answers: list) -> float:
    """Return the most common answer (rounded to nearest 0.5)."""
    # Round to nearest integer to cluster near-identical answers
    rounded = [round(a) for a in answers if a is not None]
    if not rounded:
        return None
    return float(collections.Counter(rounded).most_common(1)[0][0])


N_SC = 8  # samples for self-consistency

print(f'Self-consistency (majority vote over N={N_SC} samples):')
print(f'{"Problem":<25} {"Gold":>5}  {"Single":>7}  {"MajVote":>8}')
print('-' * 55)

single_correct = 0
mv_correct = 0

for question, gold in EVAL_PROBLEMS:
    prompt = f'Q: {question}\nA: The answer is'

    # Single sample
    single_resp = decode_topp(prompt, p=0.95, temp=0.8, max_new=20)
    single_pred = extract_answer(single_resp)
    single_ok   = reward(single_resp, gold)

    # Majority vote
    preds = [extract_answer(decode_topp(prompt, p=0.95, temp=0.9, max_new=20))
             for _ in range(N_SC)]
    mv_pred = majority_vote(preds)
    mv_ok   = 1.0 if (mv_pred is not None and abs(mv_pred - gold) <= 0.5) else 0.0

    single_correct += single_ok
    mv_correct     += mv_ok
    print(f'{question:<25} {gold:>5}  {str(single_pred):>7}  {str(mv_pred):>8}')

print(f'\nSingle-sample accuracy: {single_correct}/{len(EVAL_PROBLEMS)}')
print(f'Majority-vote accuracy:  {mv_correct}/{len(EVAL_PROBLEMS)}')

## 6. Rejection Sampling with Threshold τ

In [ ]:
def rejection_sample_until(question: str, gold: float, tau: float = 0.5, max_attempts: int = 16):
    """Sample until reward > tau or max_attempts exhausted."""
    prompt = f'Q: {question}\nA: The answer is'
    for attempt in range(1, max_attempts + 1):
        resp = decode_topp(prompt, p=0.95, temp=0.9, max_new=20)
        r = reward(resp, gold)
        if r >= tau:
            return {'response': resp, 'attempts': attempt, 'reward': r, 'success': True}
    return {'response': resp, 'attempts': max_attempts, 'reward': r, 'success': False}


print('Rejection sampling with tau=1.0 (exact correctness required):')
for question, gold in EVAL_PROBLEMS[:4]:
    result = rejection_sample_until(question, gold, tau=1.0, max_attempts=16)
    print(f'  {question:<25} | attempts={result["attempts"]:2d} | success={result["success"]} '
          f'| response={result["response"][:30]}')

# Theoretical expected attempts = 1/p_accept
print(f'\nTheoretical E[attempts] = 1/p = 1/{p_single:.2f} = {1/p_single:.1f}')

## 7. MCTS for LLMs — Simplified 2-Level Tree Search

### 7a. Concept

```
Root: [prompt tokens]
  ├── Expand: sample top-3 next tokens from policy
  │     ├── Token A → generate completion → score with verifier → value V_A
  │     ├── Token B → generate completion → score with verifier → value V_B
  │     └── Token C → generate completion → score with verifier → value V_C
  └── Select: choose token with highest value; backpropagate V to root
```

### 7b. Implementation

In [ ]:
def mcts_2level(question: str, gold: float, top_k_expand: int = 3, rollout_len: int = 20) -> dict:
    """2-level MCTS: expand top_k_expand next tokens, score each rollout, return best."""
    prompt = f'Q: {question}\nA: The answer is'
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    prompt_len = enc['input_ids'].shape[1]

    # Get next-token distribution from root
    with torch.no_grad():
        root_logits = model(**enc).logits[0, -1, :]
    root_probs = F.softmax(root_logits, dim=-1)
    top_ids = torch.topk(root_probs, top_k_expand).indices.tolist()

    node_values = []
    node_texts  = []

    for token_id in top_ids:
        # Build child sequence: prompt + chosen token
        child_ids = torch.cat(
            [enc['input_ids'], torch.tensor([[token_id]]).to(DEVICE)], dim=1
        )
        # Rollout from child
        with torch.no_grad():
            out = model.generate(
                child_ids, max_new_tokens=rollout_len, temperature=0.7,
                do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
        new_tokens = out[0][prompt_len:]
        rollout_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        value = reward(rollout_text, gold)
        node_values.append(value)
        node_texts.append(rollout_text)

    best_idx  = int(np.argmax(node_values))
    root_value = float(np.mean(node_values))  # backpropagate average

    return {
        'best_text':   node_texts[best_idx],
        'best_value':  node_values[best_idx],
        'root_value':  root_value,
        'all_values':  node_values,
        'all_texts':   node_texts,
    }


print('MCTS 2-level search:')
for question, gold in EVAL_PROBLEMS[:3]:
    result = mcts_2level(question, gold)
    print(f'  {question}')
    print(f'    node values  = {[round(v, 2) for v in result["all_values"]]}')
    print(f'    best output  = {result["best_text"][:55]}')
    print(f'    root value   = {result["root_value"]:.2f}')
    print()

## 8. Trade-off Plot: Quality vs Compute Budget

In [ ]:
# Estimate quality and compute cost for each strategy
# Compute cost is measured as multiples of single-forward-pass cost

strategies_summary = [
    ('Greedy',          1,   p_single * 0.9),   # deterministic, slightly below avg
    ('Temp sampling',   1,   p_single),
    ('Top-k (k=50)',    1,   p_single * 1.02),
    ('Top-p (p=0.9)',   1,   p_single * 1.03),
    ('Best-of-4',       4,   1-(1-p_single)**4),
    ('Best-of-8',       8,   1-(1-p_single)**8),
    ('Self-cons N=8',   8,   min(1-(1-p_single)**8 * 1.05, 1.0)),  # slight majority-vote gain
    ('MCTS (k=3)',      3,   1-(1-p_single)**3 * 1.1),  # structured search
]

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(strategies_summary)))
for (name, cost, quality), color in zip(strategies_summary, colors):
    ax.scatter(cost, min(quality, 1.0), s=150, color=color, zorder=3, label=name)
    ax.annotate(name, (cost, min(quality, 1.0)), textcoords='offset points',
                xytext=(6, 4), fontsize=8)

ax.set_xlabel('Relative compute cost (× single-sample)')
ax.set_ylabel('Estimated accuracy')
ax.set_title('Quality vs Compute Budget for Inference Strategies')
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=7)
plt.tight_layout()
plt.savefig('ttc_quality_vs_compute.png', dpi=120)
plt.show()

## 9. Summary

| Strategy | Compute | Quality gain | When to use |
|---|---|---|---|
| Greedy | 1× | Baseline | Deterministic, latency-critical |
| Temperature/Top-k/Top-p | 1× | Comparable to greedy | Diversity needed |
| Best-of-N | N× | `1-(1-p)^N` | When a verifier is available |
| Self-consistency | N× | Slight gain over best-of-N | No verifier; majority vote |
| Rejection sampling | ~N/p × | Guaranteed correct or fail | High-quality requirement |
| MCTS | Tree× | Guided search with backprop | Complex multi-step reasoning |

All strategies improve quality at the cost of more computation — the trade-off curve is the key design decision.

**Chapter 14** explores how to get the model to *self-improve* through Constitutional AI and self-play.